In [ ]:
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim

In [ ]:
df = pd.read_csv('weatherAUS.csv')

In [ ]:
df['Date'] = pd.DatetimeIndex(df['Date'])

In [ ]:
from utils import remove_records_by_year
years_to_delete = [2007, 2008, 2017]
df = remove_records_by_year(df, 'Date', years_to_delete)

In [ ]:
df['Week'] = df['Date'].dt.isocalendar().week

In [ ]:
from utils import split_city_name
df_location = pd.DataFrame(df['Location'].drop_duplicates())
df_location.reset_index(drop=True, inplace=True)
df_location.sort_values(by='Location', ascending=True, inplace=True)
df_location['Location_split'] = df_location['Location'].apply(split_city_name)

In [ ]:
df_location['Latitude'] = None
df_location['Longitude'] = None
df_location['State'] = None

In [ ]:
geolocator = Nominatim(user_agent="AUS_geocoding")

In [ ]:
from utils import get_lat_lon
df_location[['Latitude', 'Longitude']] = df_location['Location_split'].apply(get_lat_lon).apply(pd.Series)

In [ ]:
from utils import get_state
df_location['State'] = df_location.apply(lambda row: get_state(row['Latitude'], row['Longitude'], row['Location_split']), axis=1)

In [ ]:
df = pd.merge(df, df_location, on='Location', how='left')

In [ ]:
from utils import remove_records_by_state
states_to_keep = ['New South Wales', 'Victoria', 'Queensland', 'Australian Capital Territory']
df = remove_records_by_state(df, 'State', states_to_keep)

In [ ]:
df = df.sort_values(by=['Location', 'Date'], axis=0, ascending=[True, True])

In [ ]:
from utils import pressure_to_hpa
df[['Pressure9am', 'Pressure3pm']] = df[['Pressure9am', 'Pressure3pm']].apply(pressure_to_hpa)

In [ ]:
from utils import impute_rain_columns
df = impute_rain_columns(df)

In [ ]:
df['RainTomorrow'] = df['RainTomorrow'].map({'No': 0, 'Yes': 1}).astype('object')
df['RainToday'] = df['RainToday'].map({'No': 0, 'Yes': 1}).astype('object')

In [ ]:
from utils import remove_columns_with_nulls
df = remove_columns_with_nulls(df, df.columns.to_list(), threshold=35)

In [ ]:
df.dropna(subset=['RainToday', 'RainTomorrow'], how='any', inplace=True)

In [ ]:
from utils import split_numeric_categorical_columns
numeric_columns, categorical_columns = split_numeric_categorical_columns(df)

In [ ]:
column_groups = {
    'Temp_columns': 'Temp',
    'WindSpeed_columns': ['Wind', 'Speed'],
    'Humidity_columns': 'Humidity',
    'Pressure_columns': 'Pressure',
    'Rainfall_columns': 'Rainfall'
}

columns_by_group = {}

for group_name, keywords in column_groups.items():
    columns_by_group[group_name] = []
    for col in numeric_columns:
        if isinstance(keywords, list):
            if all(keyword in col for keyword in keywords):
                columns_by_group[group_name].append(col)
        else:
            if keywords in col:
                columns_by_group[group_name].append(col)

In [ ]:
from utils import impute_missing_values_numerical_columns
for col_list in columns_by_group.values():
    df = impute_missing_values_numerical_columns(df, col_list)

In [ ]:
columns_to_remove = ['Rainfall', 'Latitude', 'Longitude', 'Week']

filtered_columns = []
for col in numeric_columns:
    if col not in columns_to_remove:
        filtered_columns.append(col)

In [ ]:
from utils import cap_outliers_with_iqr
df, outliers_info = cap_outliers_with_iqr(df, filtered_columns)

In [ ]:
df['TempRange'] = df['MaxTemp'] - df['MinTemp']
numeric_columns, categorical_columns = split_numeric_categorical_columns(df)

#### quedan 3 numericas por resolver

In [ ]:
(df[numeric_columns].isna().sum()*100/df[numeric_columns].shape[0]).round(2).sort_values(ascending=False)

In [ ]:
df[numeric_columns].isna().sum().sort_values(ascending=False)

In [ ]:
mask_1 = df['Pressure3pm'].isna()
mask_2 = df['Pressure9am'].isna()
mask_3 = df['WindGustSpeed'].isna()

combined_mask = mask_1 & mask_2 & mask_3

combined_mask.sum()

#### imputación y reduccion wnd_directions

In [ ]:
wind_direction_columns = []
for col in df.columns:
    if 'Wind' and 'Dir' in col:
        wind_direction_columns.append(col)

In [ ]:
unique_wind_directions = ['NNW', 'N', 'WNW', 'NW', 'WSW', 'W', 'SSW',  'SW', 
                          'SSE', 'S', 'ESE', 'SE', 'ENE', 'E', 'NNE', 'NE', np.nan]
summary_data = {}

for col in wind_direction_columns:
    value_counts = df[col].value_counts(dropna=False)
    summary_data[col] = value_counts.reindex(unique_wind_directions, fill_value=0)

summary_df = pd.DataFrame(summary_data)

summary_df['Total'] = summary_df.sum(axis=1)

total_row = summary_df.sum(axis=0)
summary_df.loc['Total'] = total_row

summary_df

In [ ]:
from utils import impute_wind_directions
df = impute_wind_directions(df, wind_direction_columns)

In [ ]:
unique_wind_directions = ['NNW', 'N', 'WNW', 'NW', 'WSW', 'W', 'SSW',  'SW', 
                          'SSE', 'S', 'ESE', 'SE', 'ENE', 'E', 'NNE', 'NE', np.nan]
summary_data = {}

for col in wind_direction_columns:
    value_counts = df[col].value_counts(dropna=False)
    summary_data[col] = value_counts.reindex(unique_wind_directions, fill_value=0)

summary_df = pd.DataFrame(summary_data)

summary_df['Total'] = summary_df.sum(axis=1)

total_row = summary_df.sum(axis=0)
summary_df.loc['Total'] = total_row

summary_df

In [ ]:
unique_wind_directions = ['N', 'NW', 'W', 'SW', 'S', 'SE', 'E', 'NE', np.nan]
summary_data = {}

for col in wind_direction_columns:
    value_counts = df[col].value_counts(dropna=False)
    summary_data[col] = value_counts.reindex(unique_wind_directions, fill_value=0)

summary_df = pd.DataFrame(summary_data)

summary_df['Total'] = summary_df.sum(axis=1)

total_row = summary_df.sum(axis=0)
summary_df.loc['Total'] = total_row

summary_df

#### VER FORMA DE IMPUTAR CATEGÓRICOS

In [ ]:
(df.isna().sum()*100/df.shape[0]).round(2).sort_values(ascending=False)

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
mask_4 = df['WindGustDir'].isna()
mask_5 = df['WindGustSpeed'].isna()

combined_mask_2 = mask_4 & mask_5

combined_mask_2.sum()

In [ ]:
df.dropna(subset=['WindGustSpeed', 'WindGustDir'], how='all', inplace=True)

In [ ]:
df.dropna(subset=['Pressure3pm', 'Pressure9am'], how='all', inplace=True)

In [ ]:
columns_to_remove = ['Week', 'Location_split', 'Latitude', 'Longitude']
filtered_columns = []
for col in df.columns.to_list():
    if col not in columns_to_remove:
        filtered_columns.append(col)

In [ ]:
df = df[filtered_columns]

# OHE State

In [ ]:
from utils import one_hot_encoding

In [ ]:
df = one_hot_encoding(df, 'State')
df = one_hot_encoding(df, wind_direction_columns)